In [13]:
from email import policy
from email.parser import BytesParser
from html.parser import HTMLParser
from pathlib import Path
import pickle

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

In [14]:
current_dir = Path.cwd().resolve()
project_root = None
corpus_root = None

for candidate in (current_dir, *current_dir.parents):
    nested_corpus = candidate / "notebook"
    if (nested_corpus / "easy_ham").is_dir() and (nested_corpus / "spam").is_dir():
        project_root = candidate
        corpus_root = nested_corpus
        break
    if (candidate / "easy_ham").is_dir() and (candidate / "spam").is_dir():
        corpus_root = candidate
        project_root = candidate.parent if candidate.name.lower() == "notebook" else candidate
        break

if corpus_root is None:
    raise FileNotFoundError(
        "Could not find easy_ham/ and spam/ folders. Extract both SpamAssassin archives into the project notebook/ folder."
    )

ham_dir = corpus_root / "easy_ham"
spam_dir = corpus_root / "spam"
print(f"Corpus folder: {corpus_root}")
print(f"Project folder: {project_root}")
print(f"Ham files: {sum(path.is_file() for path in ham_dir.rglob('*')):,}")
print(f"Spam files: {sum(path.is_file() for path in spam_dir.rglob('*')):,}")

Corpus folder: C:\Users\Akshat\Desktop\spamsys\notebook
Project folder: C:\Users\Akshat\Desktop\spamsys
Ham files: 2,551
Spam files: 501


In [15]:
class _HTMLTextExtractor(HTMLParser):
    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.parts = []
        self.hidden_depth = 0

    def handle_starttag(self, tag, attrs):
        if tag.lower() in {"script", "style"}:
            self.hidden_depth += 1

    def handle_endtag(self, tag):
        if tag.lower() in {"script", "style"} and self.hidden_depth:
            self.hidden_depth -= 1

    def handle_data(self, data):
        if not self.hidden_depth:
            self.parts.append(data)


def _decode_part(part):
    payload = part.get_payload(decode=True)
    if payload is None:
        content = part.get_payload()
        return content if isinstance(content, str) else ""

    charset = part.get_content_charset() or "utf-8"
    try:
        return payload.decode(charset, errors="replace")
    except LookupError:
        return payload.decode("utf-8", errors="replace")


def extract_email_text(email_path):
    message = BytesParser(policy=policy.default).parsebytes(email_path.read_bytes())
    subject = str(message.get("subject", "")).strip()
    plain_parts = []
    html_parts = []

    for part in message.walk():
        if part.is_multipart() or part.get_content_disposition() == "attachment":
            continue

        content_type = part.get_content_type()
        if content_type not in {"text/plain", "text/html"}:
            continue

        content = _decode_part(part).strip()
        if content_type == "text/plain" and content:
            plain_parts.append(content)
        elif content_type == "text/html" and content:
            html_parts.append(content)

    if plain_parts:
        body = "\n".join(plain_parts)
    elif html_parts:
        parser = _HTMLTextExtractor()
        for html in html_parts:
            parser.feed(html)
        body = " ".join(parser.parts)
    else:
        body = ""

    return f"Subject: {subject}\n{body}".strip()

In [16]:
records = []
skipped_files = 0

for label, folder in ((0, ham_dir), (1, spam_dir)):
    for email_path in sorted(folder.rglob("*")):
        if not email_path.is_file():
            continue
        try:
            text = extract_email_text(email_path)
        except (OSError, ValueError, TypeError):
            skipped_files += 1
            continue
        if text.strip():
            records.append({"label": label, "text": text})
        else:
            skipped_files += 1

emails = pd.DataFrame(records, columns=["label", "text"])
print(f"Parsed emails: {len(emails):,}")
print(f"Skipped unreadable or empty files: {skipped_files:,}")
print("\nClass counts (0 = ham, 1 = spam):")
print(emails["label"].value_counts().sort_index())
emails.head(3)

Parsed emails: 3,052
Skipped unreadable or empty files: 0

Class counts (0 = ham, 1 = spam):
label
0    2551
1     501
Name: count, dtype: int64


,label,text
0,0,Subject: Re: New Sequences Window\nDate: ...
1,0,Subject: [zzzzteana] RE: Alexander\nMartin A p...
2,0,Subject: [zzzzteana] Moscow bomber\nMan Threat...


In [17]:
emails = emails[emails["text"].str.strip().ne("")].copy()
before_deduplication = len(emails)
emails = emails.drop_duplicates(subset="text").reset_index(drop=True)

print(f"Exact duplicate emails removed: {before_deduplication - len(emails):,}")
print("Class counts after cleanup (0 = ham, 1 = spam):")
print(emails["label"].value_counts().sort_index())

Exact duplicate emails removed: 171
Class counts after cleanup (0 = ham, 1 = spam):
label
0    2409
1     472
Name: count, dtype: int64


In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    emails["text"],
    emails["label"],
    test_size=0.2,
    random_state=42,
    stratify=emails["label"],
)

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, lowercase=True)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model = MultinomialNB(class_prior=[0.5, 0.5])
model.fit(X_train_tfidf, y_train)
print(f"Training emails: {len(X_train):,}")
print(f"Test emails: {len(X_test):,}")
print("Training complete.")

Training emails: 2,304
Test emails: 577
Training complete.


In [19]:
predictions = model.predict(X_test_tfidf)
print(f"Accuracy: {accuracy_score(y_test, predictions):.4f}")
print("\nConfusion matrix (rows = actual, columns = predicted):")
confusion = confusion_matrix(y_test, predictions, labels=[0, 1])
display(
    pd.DataFrame(
        confusion,
        index=["Actual ham", "Actual spam"],
        columns=["Predicted ham", "Predicted spam"],
    )
)
print("\nClassification report:")
print(
    classification_report(
        y_test,
        predictions,
        labels=[0, 1],
        target_names=["Ham", "Spam"],
        zero_division=0,
    )
)

Accuracy: 0.9324

Confusion matrix (rows = actual, columns = predicted):


,Predicted ham,Predicted spam
Actual ham,482,0
Actual spam,39,56



Classification report:
              precision    recall  f1-score   support

         Ham       0.93      1.00      0.96       482
        Spam       1.00      0.59      0.74        95

    accuracy                           0.93       577
   macro avg       0.96      0.79      0.85       577
weighted avg       0.94      0.93      0.92       577



In [20]:
email_model_path = project_root / "email_model.pkl"
email_vectorizer_path = project_root / "email_vectorizer.pkl"

with email_model_path.open("wb") as file:
    pickle.dump(model, file)

with email_vectorizer_path.open("wb") as file:
    pickle.dump(vectorizer, file)

print(f"Saved model: {email_model_path}")
print(f"Saved vectorizer: {email_vectorizer_path}")

Saved model: C:\Users\Akshat\Desktop\spamsys\email_model.pkl
Saved vectorizer: C:\Users\Akshat\Desktop\spamsys\email_vectorizer.pkl


In [21]:
sample_email = """Subject: Account notice
Please review the attached account update when you have a moment."""
sample_features = vectorizer.transform([sample_email])
sample_prediction = int(model.predict(sample_features)[0])
label_name = "Spam" if sample_prediction == 1 else "Ham (not spam)"
print(f"Prediction: {label_name}")

Prediction: Ham (not spam)
